In [6]:
# Experiment: Bayesian Network for Diagnosis of Heart Patients
# 1. Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

# 2. Load dataset
df = pd.read_csv("heart_disease.csv")
print("First five records:")
print(df.head())
print("\nDataset Shape:", df.shape)

# 3. Data preprocessing
# Convert Yes/No into 1/0
df["disease"] = df["heartdisease"].str.strip().map({
    "No": 0,
    "Yes": 1
})

# Convert numerical values into categories
data = pd.DataFrame({
    "age": pd.cut(
        df["age"],
        bins=[0, 45, 55, 100],
        labels=[0, 1, 2]
    ),

    "sex": df["sex"],
    "cp": df["cp"],

    "bp": pd.cut(
        df["trestbps"],
        bins=[0, 120, 140, 300],
        labels=[0, 1, 2]
    ),

    "chol": pd.cut(
        df["chol"],
        bins=[0, 200, 240, 700],
        labels=[0, 1, 2]
    ),

    "fbs": df["fbs"],
    "restecg": df["restecg"],

    "thalach": pd.cut(
        df["thalach"],
        bins=[0, 120, 150, 250],
        labels=[0, 1, 2]
    ),

    "exang": df["exang"],

    "oldpeak": pd.cut(
        df["oldpeak"],
        bins=[-0.1, 0.5, 2, 10],
        labels=[0, 1, 2]
    ),
    "slope": df["slope"],
    "ca": df["ca"],
    "thal": df["thal"],
    "disease": df["disease"]
})

# Remove missing values and convert to integers
data = data.dropna().astype(int)
print("\nPreprocessed Data:")
print(data.head())

# 4. Split dataset
train, test = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["disease"]
)
print("\nTraining Records:", len(train))
print("Testing Records:", len(test))

# 5. Create Bayesian Network
edges = [
    ("age", "bp"),
    ("age", "chol"),
    ("age", "disease"),
    ("sex", "disease"),
    ("bp", "disease"),
    ("chol", "disease"),
    ("fbs", "disease"),
    ("restecg", "disease"),
    ("disease", "cp"),
    ("disease", "exang"),
    ("disease", "thalach"),
    ("disease", "oldpeak"),
    ("disease", "slope"),
    ("disease", "ca"),
    ("disease", "thal")
]
model = DiscreteBayesianNetwork(edges)

# 6. Learn Conditional Probability Distributions (CPDs)
# Define all possible states
state_names = {
    col: sorted(data[col].unique().tolist())
    for col in data.columns
}

estimator = BayesianEstimator(
    model,
    train,
    state_names=state_names
)

cpds = estimator.get_parameters(
    prior_type="BDeu",
    equivalent_sample_size=10
)

model.add_cpds(*cpds)
print("\nModel Valid:", model.check_model())
print("\nConditional Probability Table:")
print(model.get_cpds("cp"))

# 7. Perform inference for two patients
infer = VariableElimination(model)

# Patient 1: Older male with asymptomatic chest pain
q1 = infer.query(
    variables=["disease"],
    evidence={
        "age": 2,
        "sex": 1,
        "cp": 4,
        "exang": 1
    }
)
print("\nDiagnosis of Patient 1:")
print(q1)

# Patient 2: Young female with non-anginal chest pain

q2 = infer.query(
    variables=["disease"],
    evidence={
        "age": 0,
        "sex": 0,
        "cp": 3,
        "exang": 0,
        "thalach": 2
    }
)
print("\nDiagnosis of Patient 2:")
print(q2)

# 8. Test the model
evidence_cols = [
    "age", "sex", "cp", "bp", "chol",
    "fbs", "restecg", "thalach", "exang",
    "oldpeak", "slope", "ca", "thal"
]
predictions = []

for _, row in test.iterrows():

    result = infer.query(
        variables=["disease"],
        evidence=row[evidence_cols].to_dict(),
        show_progress=False
    )
    prediction = int(result.values[1] >= result.values[0])
    predictions.append(prediction)

# 9. Calculate accuracy
actual = test["disease"]
accuracy = accuracy_score(actual, predictions)
print("\nModel Accuracy:", round(accuracy * 100, 2), "%")

# 10. Confusion Matrix and Classification Report
print("\nConfusion Matrix:")
print(confusion_matrix(actual, predictions))
print("\nClassification Report:")
print(
    classification_report(
        actual,
        predictions,
        target_names=["No Disease", "Disease"]
    )
)

First five records:
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal heartdisease  
0    3.0  0.0   6.0           No  
1    2.0  3.0   3.0          Yes  
2    2.0  2.0   7.0          Yes  
3    3.0  0.0   3.0           No  
4    1.0  0.0   3.0           No  

Dataset Shape: (297, 14)

Preprocessed Data:
   age  sex  cp  bp  chol  fbs  restecg  thalach  exang  oldpeak  slope  ca  \
0    2    1   1   2     1    1        2        1      0        2      3   0   
1    2    1   4   2     2    0        2        0      1        1      2   3   
2    2    1   4   0     

C:\Users\20pri\AppData\Local\Temp\ipykernel_44532\1507061536.py:110: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(
